# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssarimm/Machine-Learning-Internship---Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Content age and decline

The paper reports that growing content averaged 184 days old, while declining content averaged 230 days old. It also reports that growing pages were longer on average than declining pages (3,180 versus 2,311 words). The paper describes these as observational comparisons rather than causal evidence.

**My methodology question:** The paper defines trend direction using the impression change between the most recent 30 days and the preceding 30 days. Were content age and word count measured independently of this outcome window, and were other factors that could explain the differences considered? This would help clarify whether age and content depth are useful indicators of decline or simply characteristics associated with the observed groups.

### Finding 2: Logistic regression for growth and decline

The paper's exploratory ML appendix reports 71% holdout accuracy for a logistic regression model distinguishing growing from declining pages. It describes content age as a strong negative signal for growth and days visible and recent impressions as positive signals. The paper cautions that these are descriptive indicators, not direct instructions to optimize individual variables.

**My methodology question:** How was the holdout set constructed? In particular, were pages from the same brand kept together, or could the same brands appear in both training and testing? Also, because trend direction is calculated from recent impression changes, how were the feature measurement windows checked to ensure that recent impressions did not overlap with the period used to define the label? These details would help assess how well the reported accuracy might generalize to unseen brands or future periods.

### Why this audit matters

These questions are intended to understand the scope of the evidence, not to dismiss the findings. Grouped validation and careful separation of feature and label time windows can help establish how reliably a model generalizes beyond the data used to build it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

In [4]:

from sklearn.model_selection import GroupShuffleSplit

# Keep rows that have both a client ID and a target label
audit_df = df.dropna(subset=["client_id", "trend_direction"]).copy()

# Features used in the Week 5 model
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "days_since_last_update"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level"
]

features = numeric_features + categorical_features

X_audit = audit_df[features]
y_audit = (audit_df["trend_direction"] == "down").astype(int)
groups_audit = audit_df["client_id"]

# Keep each client's pages entirely in train OR test
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X_audit, y_audit, groups=groups_audit)
)

train_clients = set(groups_audit.iloc[train_idx])
test_clients = set(groups_audit.iloc[test_idx])

print("Total rows:", len(audit_df))
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Clients appearing in BOTH sets:", len(train_clients & test_clients))
print("Training decline rate:", round(y_audit.iloc[train_idx].mean(), 3))
print("Test decline rate:", round(y_audit.iloc[test_idx].mean(), 3))


Total rows: 30000
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Clients appearing in BOTH sets: 0
Training decline rate: 0.55
Test decline rate: 0.511


### Grouped train/test split audit

The dataset contains 30,000 rows. The grouped split allocated 23,837 rows to training and 6,163 rows to testing. The training set contains 25 clients, while the test set contains 7 clients.

There are **zero clients shared between the training and test sets**, so the model is evaluated on clients it did not see during training. The decline rate is 55.0% in training and 51.1% in testing, indicating a modest difference in target distribution between the two sets.

This is a stronger evaluation than randomly splitting individual pages when pages from the same client may be similar. However, grouping by client does not eliminate potential temporal leakage if feature measurement windows overlap with the target measurement window.


In [5]:

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Use the same feature set and grouped split
X_train = X_audit.iloc[train_idx].copy()
X_test = X_audit.iloc[test_idx].copy()
y_train = y_audit.iloc[train_idx].copy()
y_test = y_audit.iloc[test_idx].copy()

# Prepare numeric and categorical features
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Decision Tree model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=30,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Probability that a page is declining
model_scores = model.predict_proba(X_test)[:, 1]

# Simple baseline: stale content gets 2 points;
# high search demand gets 1 point.
stale_score = (
    X_test["days_since_last_update"].fillna(-1) >= 180
).astype(int) * 2

positive_search_volume = X_train.loc[
    X_train["search_volume"] > 0, "search_volume"
].dropna()

demand_threshold = positive_search_volume.median()

high_demand_score = (
    X_test["search_volume"].fillna(0) >= demand_threshold
).astype(int)

baseline_scores = stale_score + high_demand_score

# Evaluate both rankings
results = pd.DataFrame({
    "Model": ["Decision Tree", "Simple baseline"],
    "ROC-AUC": [
        roc_auc_score(y_test, model_scores),
        roc_auc_score(y_test, baseline_scores)
    ],
    "Average Precision": [
        average_precision_score(y_test, model_scores),
        average_precision_score(y_test, baseline_scores)
    ]
})

print("Baseline search-volume threshold:", demand_threshold)
display(results.round(3))


Baseline search-volume threshold: 20.0


,Model,ROC-AUC,Average Precision
0,Decision Tree,0.508,0.510
1,Simple baseline,0.486,0.504


### Model performance under a client-grouped split

The Decision Tree achieved a ROC-AUC of 0.508 and Average Precision (AP) of 0.510 on the held-out test set. The simple baseline achieved a ROC-AUC of 0.486 and AP of 0.504.

The Decision Tree performed slightly better than the baseline on both metrics, but the improvement was small. Its AP was also approximately equal to the test-set decline rate of 0.511, suggesting limited ranking value beyond the prevalence of declining pages.

These results do not support a strong claim that the model reliably identifies declining content. Further validation is needed, including a leakage audit and, if the available data permits it, evaluation using features measured strictly before the target period.


In [6]:

# Inspect columns that may overlap with the target's time window
time_related_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in [
        "date", "period", "window", "impression",
        "click", "pageview", "session", "trend"
    ])
]

print("Potentially time-related columns:")
for col in time_related_columns:
    print("-", col)


Potentially time-related columns:
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- engaged_sessions_90d
- ai_sessions_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- days_since_last_update
- impression_tier
- trend_direction
- trend_pct
- is_initial_refresh_candidate


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:

# Features currently used by our Section 2 audit model
print("Features used by the model:")
print(features)

# Columns that need special attention during the leakage audit
risk_columns = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "days_with_impressions",
    "impression_tier",
    "is_initial_refresh_candidate"
]

print("\nLeakage audit checklist:")
for col in risk_columns:
    if col not in df.columns:
        status = "Not present"
    elif col in features:
        status = "USED as a feature — investigate"
    else:
        status = "Not used by this audit model"

    print(f"{col}: {status}")


Features used by the model:
['search_volume', 'competition', 'cpc', 'word_count', 'content_age_days', 'days_since_last_update', 'content_type', 'main_intent', 'competition_level']

Leakage audit checklist:
trend_direction: Not used by this audit model
trend_pct: Not used by this audit model
impressions_last_30d: Not used by this audit model
impressions_prev_30d: Not used by this audit model
clicks_last_30d: Not used by this audit model
clicks_prev_30d: Not used by this audit model
sessions_last_30d: Not used by this audit model
sessions_prev_30d: Not used by this audit model
impressions_90d: Not used by this audit model
clicks_90d: Not used by this audit model
pageviews_90d: Not used by this audit model
sessions_90d: Not used by this audit model
days_with_impressions: Not used by this audit model
impression_tier: Not used by this audit model
is_initial_refresh_candidate: Not used by this audit model


### Leakage audit findings

The audited Decision Tree uses nine features: six numeric features (`search_volume`, `competition`, `cpc`, `word_count`, `content_age_days`, and `days_since_last_update`) and three categorical features (`content_type`, `main_intent`, and `competition_level`).

The model does not directly use `trend_direction`, `trend_pct`, the recent/previous 30-day performance metrics, the 90-day analytics aggregates, `impression_tier`, or `is_initial_refresh_candidate`.

This reduces the risk of direct leakage through the excluded columns. However, the feature definitions and measurement timestamps have not yet been verified, so temporal leakage cannot be ruled out. The grouped client split prevents client overlap between training and test sets, but it does not establish that every feature was available before the prediction period.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Overstated claim:** “The model can reliably predict which content pages will decline and identify pages that should be refreshed.”

**Evidence-aligned claim:** “In a client-grouped holdout evaluation, the Decision Tree achieved a ROC-AUC of 0.508 and Average Precision of 0.510, only slightly outperforming a simple baseline (ROC-AUC 0.486; AP 0.504). These results provide limited evidence of useful ranking ability. The model should not yet be used to make automated refresh decisions. Further validation, including verification of feature timing and evaluation on genuinely future outcomes, is needed.”


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.